In [ ]:
!pip install kaggle

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mahdimashayekhi/disease-risk-from-daily-habits")

print("Path to dataset files:", path)

100%|██████████| 20.8M/20.8M [00:01<00:00, 12.1MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/mahdimashayekhi/disease-risk-from-daily-habits/versions/1


In [ ]:
!pip install catboost

In [ ]:
#import library
import pandas as pd
import os

# Lihat isi folder dataset
print(os.listdir(path))

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from catboost import CatBoostClassifier

In [ ]:
df = pd.read_csv(path + "/health_lifestyle_classification.csv")
df.head()

In [ ]:
df.info()

In [ ]:
pd.set_option('display.max_columns', None)
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df = df.drop(columns=[
'survey_code',
'bmi_estimated',
'bmi_scaled',
'bmi_corrected',
'mental_health_score',
'mental_health_support',
'education_level',
'job_type',
'occupation',
'income',
'healthcare_access',
'insurance',
'sunlight_exposure',
'family_history',
'pet_owner',
'electrolyte_level',
'gene_marker_flag',
'environmental_risk_score',
'daily_supplement_dosage',
'device_usage',
'exercise_type',
'diet_type',
'caffeine_intake',
'sleep_quality',
'alcohol_consumption',
'smoking_level'
])

df.columns

In [ ]:
num_cols = df.select_dtypes(include=['int64','float64']).columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())


cat_cols = df.select_dtypes(include='object').columns

for col in cat_cols:
    df[col] = df[col].fillna("Unknown")

In [ ]:
df.isnull().sum()

In [ ]:
df['target'] = df['target'].map({
    'healthy':0,
    'diseased':1
})



df['target'].value_counts()

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

In [ ]:
df.isnull().sum()

In [ ]:
# BMI category (lebih informatif daripada angka)
df['bmi_category'] = pd.cut(df['bmi'],
                           bins=[0,18.5,25,30,100],
                           labels=['underweight','normal','overweight','obese'])

# Pola gaya hidup
df['high_sugar'] = (df['sugar_intake'] > 50).astype(int)
df['low_steps'] = (df['daily_steps'] < 5000).astype(int)
df['high_stress'] = (df['stress_level'] > 7).astype(int)

# Rasio penting
df['calorie_per_meal'] = df['calorie_intake'] / (df['meals_per_day'] + 1)
df['activity_ratio'] = df['daily_steps'] / (df['calorie_intake'] + 1)

df['cardio_risk'] = (
    (df['blood_pressure'] > 130).astype(int) +
    (df['cholesterol'] > 200).astype(int) +
    (df['glucose'] > 120).astype(int)
)

df['bad_lifestyle'] = (
    (df['sleep_hours'] < 6).astype(int) +
    (df['stress_level'] > 7).astype(int) +
    (df['screen_time'] > 8).astype(int)
)

df['activity_balance'] = df['daily_steps'] / (df['calorie_intake'] + 1)



In [ ]:
X = df.drop(columns=drop_cols + ['target'])
y = df['target']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()


In [ ]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=4000,
    learning_rate=0.02,
    depth=8,
    l2_leaf_reg=15,
    random_strength=3,
    auto_class_weights='Balanced',
    eval_metric='AUC',
    random_state=42,
    verbose=200
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_test, y_test),
    early_stopping_rounds=300
)

In [ ]:
from sklearn.metrics import roc_auc_score

y_prob = model.predict_proba(X_test)[:,1]
y_pred = (y_prob > 0.2).astype(int)
print(classification_report(y_test, y_pred))

print("ROC AUC:", roc_auc_score(y_test, y_prob))

In [ ]:
from sklearn.metrics import f1_score

best_f1 = 0
best_threshold = 0

for t in np.arange(0.2, 0.8, 0.05):  # jangan terlalu rendah
    y_pred = (y_prob > t).astype(int)
    score = f1_score(y_test, y_pred)

    if score > best_f1:
        best_f1 = score
        best_threshold = t

print("Best threshold:", best_threshold)
print("Best F1:", best_f1)